# 02 — Post-Metric EDA + Hypothesis Test

**Purpose:** Compare High vs Low retail buckets on computed metrics,
embed Mann-Whitney U results, and write a plain-language interpretation.

**Done condition:** ≥1 chart per metric + plain-language interpretation paragraph.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from pathlib import Path

DB_PATH = Path('../data/retail_behavior.db')
conn = sqlite3.connect(DB_PATH)
sns.set_theme(style='whitegrid')
ALPHA = 0.05

## 1. Bucket Comparison — Metric A (Volume Spike Ratio)

In [ ]:
df = pd.read_sql('SELECT * FROM metrics_results WHERE bucket IS NOT NULL', conn)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, w in zip(axes, [3, 5]):
    sub = df[df['window_days'] == w]
    sns.boxplot(data=sub, x='bucket', y='volume_spike_ratio', ax=ax)
    ax.set_title(f'Volume Spike Ratio — ±{w}d window')
    ax.set_xlabel('Retail Bucket')
plt.tight_layout()
plt.show()

## 2. Bucket Comparison — Metric B (Realized Volatility)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, w in zip(axes, [3, 5]):
    sub = df[df['window_days'] == w]
    sns.boxplot(data=sub, x='bucket', y='realized_vol', ax=ax)
    ax.set_title(f'Realized Volatility — ±{w}d window')
    ax.set_xlabel('Retail Bucket')
plt.tight_layout()
plt.show()

## 3. Mann-Whitney U Test Results

In [ ]:
results = []
for col, label in [('volume_spike_ratio', 'Metric A'), ('realized_vol', 'Metric B')]:
    for w in [3, 5]:
        sub = df[df['window_days'] == w][['bucket', col]].dropna()
        high = sub[sub['bucket'] == 'High'][col].values
        low  = sub[sub['bucket'] == 'Low'][col].values
        if len(high) >= 2 and len(low) >= 2:
            stat, p = mannwhitneyu(high, low, alternative='two-sided')
            results.append({
                'Metric': label,
                'Window': f'±{w}d',
                'High median': round(pd.Series(high).median(), 4),
                'Low median':  round(pd.Series(low).median(), 4),
                'U stat': round(stat, 1),
                'p-value': round(p, 4),
                'Verdict': 'REJECT H0' if p < ALPHA else 'FAIL TO REJECT H0'
            })

pd.DataFrame(results)

## 4. Interpretation

*[Write plain-language interpretation here after reviewing the table above.]*

Example structure:
- State the p-value result for each combination.
- Note whether ±3d and ±5d windows agree (consistency strengthens the finding).
- Note whether Metric A and Metric B agree.
- One paragraph summary: what does the data say about the original question?